## 1. Swagger 기반 Canonical Schema 생성

조회 API에서 OpenAPI JSON을 가져와 `document_id + api_id` 단위의 API별 JSON 파일과 SQLite 레코드를 생성한다.

이번 범위에는 기본 API 정보와 요청 파라미터만 포함하며, 응답 필드 평탄화와 Scenario IR `outputs`는 포함하지 않는다.

In [1]:
import hashlib
import json
import re
import sqlite3
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, Field

SWAGGER_URL = (
    'https://developers.hmall.com/main/swagger/'
    '5f205b21-baa0-44a6-a1ac-39c4d9395d38/'
    '1093902c-01a2-4167-8103-d0615749bdef'
    '?product_api_name=OPENAPI'
)
DOCUMENT_ID = 'hmall-openapi-v1'
REQUEST_TIMEOUT_SECONDS = 30

# 현재 실행 위치에서 preprocessing.ipynb가 있는 프로젝트 디렉터리를 찾는다.
def locate_project_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / 'hyejung' / 'mini_project_2']
    for candidate in candidates:
        if (candidate / 'preprocessing.ipynb').exists():
            return candidate.resolve()
    raise FileNotFoundError('preprocessing.ipynb가 있는 프로젝트 디렉터리를 찾지 못했습니다.')

PROJECT_DIR = locate_project_dir()
OUTPUT_DIR = PROJECT_DIR / 'output'
RAW_PATH = OUTPUT_DIR / 'swagger_raw.json'
DB_PATH = OUTPUT_DIR / 'canonical_schema.db'
REPORT_PATH = OUTPUT_DIR / 'canonical_schema_report.json'
API_JSON_DIR = OUTPUT_DIR / 'canonical_schema' / DOCUMENT_ID

# API 요청 파라미터 하나의 표준 표현을 정의한다.
class CanonicalParameter(BaseModel):
    path: str
    location: Literal['header', 'path', 'query', 'cookie', 'body']
    name: str
    type: str
    required: bool = False
    requiredDeclared: bool = False
    description: str | None = None
    format: str | None = None
    enum: list[Any] | None = None
    example: Any | None = None
    sourceRef: str | None = None
    xmlName: str | None = None
    xmlPath: str | None = None
    xmlAttribute: bool = False

# API 하나의 기본정보와 요청 파라미터 목록을 정의한다.
class CanonicalApi(BaseModel):
    documentId: str
    documentTitle: str
    apiId: str
    method: str
    endpoint: str
    summary: str | None = None
    description: str | None = None
    tags: list[str] = Field(default_factory=list)
    contentType: str | None = None
    requestBodyRequired: bool = False
    parameters: list[CanonicalParameter] = Field(default_factory=list)
    source: dict[str, Any] = Field(default_factory=dict)
    warnings: list[str] = Field(default_factory=list)

# Swagger 문서 단위의 식별정보와 API 목록을 정의한다.
class CanonicalDocument(BaseModel):
    documentId: str
    documentTitle: str
    openapiVersion: str | None = None
    documentVersion: str | None = None
    sourceUrl: str
    apis: list[CanonicalApi] = Field(default_factory=list)


### 정규화 규칙

- 문서 단위 식별자는 `documentId`, Swagger `info.title`은 `documentTitle`로 저장한다.
- `operationId`를 `apiId`로 사용하고, 없으면 method와 endpoint로 fallback ID를 만든다.
- header/path/query/cookie/body를 하나의 `parameters` 배열로 통합한다.
- body 경로는 `$.body...`, header 경로는 `$.headers...`로 만든다.
- `$ref`는 `components.schemas`에서 해석하고, XML `xml.name`, `xmlPath`, `xmlAttribute`와 원본 참조 경로를 보존한다.
- `required: 'true'`처럼 문자열로 된 값을 boolean으로 변환한다.
- 응답 schema와 응답 필드는 읽거나 평탄화하지 않는다.

#### XML 파싱 함수

In [2]:
# 현재 시각을 UTC ISO 8601 문자열로 반환 : 파일 및 db의 생성/수정 시간 기록
def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

# Pydantic 모델을 JSON 저장 가능한 딕셔너리로 변환
def model_to_dict(model: BaseModel) -> dict[str, Any]:
    if hasattr(model, 'model_dump'):
        return model.model_dump(mode='json')
    return model.dict()

# boolean 또는 문자열 형태의 필수 여부 값을 boolean으로 변환 (True, False)
def coerce_bool(value: Any, default: bool = False) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {'true', '1', 'yes', 'y'}:
            return True
        if normalized in {'false', '0', 'no', 'n', ''}:
            return False
    if isinstance(value, (int, float)):
        return bool(value)
    return default

# API ID를 파일명으로 사용할 수 있는 안전한 문자열로 변환 : 특수문자를 _로 치환
def safe_filename(value: str) -> str:
    cleaned = re.sub(r'[^0-9A-Za-z가-힣._-]+', '_', value).strip('._')
    return cleaned or 'unknown'

# JSONPath 기준 경로에 property 이름을 추가 : ex> $.body.user.name
def append_path(base: str, name: str) -> str:
    if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
        return f'{base}.{name}'
    return f'{base}[{json.dumps(name, ensure_ascii=False)}]'

# 참조 위치 탐색 : Swagger 문서 내부의 JSON Pointer 참조를 실제 schema 객체로 반환
# ex> resolve_json_pointer(document, "#/components/schemas/selectItemRequest")
def resolve_json_pointer(document: dict[str, Any], ref: str) -> dict[str, Any] | None:
    if not ref.startswith('#/'):
        return None
    current: Any = document
    for token in ref[2:].split('/'):
        token = token.replace('~1', '/').replace('~0', '~')
        if not isinstance(current, dict) or token not in current:
            return None
        current = current[token]
    return current if isinstance(current, dict) else None

# 최종 schema 생성 : schema의 $ref를 재귀적으로 해석(resolve_json_pointer 활용)하고 inline 속성과 병합
def resolve_schema(schema: dict[str, Any], document: dict[str, Any], warnings: list[str], visited_refs: set[str] | None = None) -> dict[str, Any]:
    visited_refs = set() if visited_refs is None else set(visited_refs)
    if not isinstance(schema, dict) or '$ref' not in schema:
        return dict(schema) if isinstance(schema, dict) else {}
    ref = schema['$ref']
    if ref in visited_refs:
        warnings.append(f'순환 $ref: {ref}')
        return {key: value for key, value in schema.items() if key != '$ref'}
    target = resolve_json_pointer(document, ref)
    if target is None:
        warnings.append(f'해석할 수 없는 $ref: {ref}')
        return {key: value for key, value in schema.items() if key != '$ref'}
    resolved = resolve_schema(target, document, warnings, visited_refs | {ref})
    resolved.update({key: value for key, value in schema.items() if key != '$ref'})
    return resolved

# description 추출 : schema 또는 example에서 파라미터 설명을 추출
# ex> 판매상품코드(예:2015700001) => 판매상품코드
def extract_description(value: dict[str, Any]) -> str | None:
    description = value.get('description')
    if description not in (None, ''):
        return str(description)
    example = value.get('example')
    if isinstance(example, str) and '(' in example:
        candidate = example.split('(', 1)[0].strip()
        return candidate or None
    return None

# 데이터타입 정규화 : Swagger 타입을 Canonical Schema의 표준 타입과 format으로 정규화
def normalize_type(schema: dict[str, Any], warnings: list[str], path: str) -> tuple[str, str | None]:
    raw_type = schema.get('type')
    schema_format = schema.get('format')
    if raw_type is None:
        if 'properties' in schema:
            raw_type = 'object'
        elif 'items' in schema:
            raw_type = 'array'
        else:
            raw_type = 'string'
            warnings.append(f'type 누락으로 string 처리: {path}')
    normalized = str(raw_type).lower()
    if normalized in {'date', 'datetime'}:
        return 'string', schema_format or normalized
    if normalized == 'date-time':
        return 'string', schema_format or 'date-time'
    if normalized not in {'string', 'integer', 'number', 'boolean', 'object', 'array'}:
        warnings.append(f'비표준 type을 string 처리: {raw_type} ({path})')
        return 'string', schema_format
    return normalized, schema_format

# xml 이름 추출 : schema에서 XML 이름과 속성 여부를 추출하고 이름이 없으면 property 이름을 사용
# ex> "xml": {"name": "Dataset"} -> Dataset
def get_xml_metadata(schema: dict[str, Any], fallback_name: str | None = None) -> tuple[str | None, bool]:
    xml = schema.get('xml') if isinstance(schema.get('xml'), dict) else {}
    xml_name = xml.get('name') or fallback_name
    return (str(xml_name) if xml_name else None, coerce_bool(xml.get('attribute')))

# xmlPath segment 생성 : XML object의 id 예시를 이용해 Dataset처럼 반복되는 요소를 식별 가능한 경로로 생성
# ex> Dataset[@id="dsCond"]
def get_xml_segment(schema: dict[str, Any], fallback_name: str, document: dict[str, Any], warnings: list[str]) -> str:
    resolved = resolve_schema(schema, document, warnings)
    xml_name, _ = get_xml_metadata(resolved, fallback_name)
    segment = xml_name or fallback_name
    properties = resolved.get('properties')
    if isinstance(properties, dict) and isinstance(properties.get('id'), dict):
        id_schema = resolve_schema(properties['id'], document, warnings)
        id_name, id_attribute = get_xml_metadata(id_schema, 'id')
        id_example = id_schema.get('example')
        if id_attribute and id_name and isinstance(id_example, str) and re.fullmatch(r'[A-Za-z_][A-Za-z0-9_-]*', id_example):
            segment = f'{segment}[@{id_name}={json.dumps(id_example, ensure_ascii=False)}]'
    return segment

# xmlPath 완성 : 현재 XML 경로에 schema 요소 또는 속성 이름을 추가
# ex> [base = /Root/Dataset[@id="dsCond"]/rows/row] + [name = slitmCd] => /Root/Dataset[@id="dsCond"]/rows/row/slitmCd
# attribute인 경우 : /Root/Dataset[@id=\"dsCond\"]/@id
def append_xml_path(base: str, schema: dict[str, Any], fallback_name: str, document: dict[str, Any], warnings: list[str]) -> str:
    resolved = resolve_schema(schema, document, warnings)
    xml_name, xml_attribute = get_xml_metadata(resolved, fallback_name)
    segment = f'@{xml_name}' if xml_attribute and xml_name else (xml_name or fallback_name)
    return f'{base}/{segment}' if base else f'/{segment}'

# CanonicalParameter 객체 생성(필드 하나 대상) : 정규화된 schema 정보를 CanonicalParameter 객체로 만들어 목록에 추가
def add_parameter(parameters: list[CanonicalParameter], schema: dict[str, Any], path: str, location: str, name: str, required: bool, required_declared: bool, source_ref: str | None, warnings: list[str], xml_path: str | None = None) -> None:
    normalized_type, schema_format = normalize_type(schema, warnings, path)
    xml_name, xml_attribute = get_xml_metadata(schema, name)
    parameters.append(CanonicalParameter(path=path, location=location, name=name, type=normalized_type, required=required, requiredDeclared=required_declared, description=extract_description(schema), format=schema_format, enum=schema.get('enum'), example=schema.get('example'), sourceRef=source_ref, xmlName=xml_name, xmlPath=xml_path, xmlAttribute=xml_attribute))

# 해석된 schema를 parameter로 평탄화 : 중첩된 object와 array schema를 재귀 순회해 요청 파라미터와 XML 경로를 평탄화한다.
# 처리 단위 : 하나의 requestBody $ref 기준으로 처리
def walk_schema(schema: dict[str, Any], base_path: str, location: str, document: dict[str, Any], parameters: list[CanonicalParameter], warnings: list[str], required: bool = False, required_declared: bool = False, name: str | None = None, source_ref: str | None = None, xml_path: str | None = None) -> None:
    if not isinstance(schema, dict):
        warnings.append(f'스키마가 object가 아님: {base_path}')
        return
    local_ref = schema.get('$ref') or source_ref
    resolved = resolve_schema(schema, document, warnings)
    normalized_type, _ = normalize_type(resolved, warnings, base_path)
    current_xml_path = xml_path
    if location == 'body' and current_xml_path is None:
        root_xml_name, _ = get_xml_metadata(resolved, name)
        current_xml_path = f'/{root_xml_name}' if root_xml_name else ''
    if normalized_type == 'object' and isinstance(resolved.get('properties'), dict):
        required_names = set(resolved.get('required') or [])
        has_required_declaration = 'required' in resolved
        for property_name, property_schema in resolved['properties'].items():
            child_xml_path = current_xml_path
            if location == 'body':
                child_resolved = resolve_schema(property_schema, document, warnings)
                _, child_is_attribute = get_xml_metadata(child_resolved, property_name)
                if child_is_attribute or not isinstance(child_resolved.get('properties'), dict):
                    child_xml_path = append_xml_path(current_xml_path or '', child_resolved, property_name, document, warnings)
                else:
                    child_segment = get_xml_segment(child_resolved, property_name, document, warnings)
                    child_xml_path = f'{current_xml_path}/{child_segment}' if current_xml_path else f'/{child_segment}'
            walk_schema(property_schema, append_path(base_path, property_name), location, document, parameters, warnings, required=property_name in required_names, required_declared=has_required_declaration, name=property_name, source_ref=local_ref, xml_path=child_xml_path)
        return
    parameter_name = name or base_path.rsplit('.', 1)[-1]
    add_parameter(parameters, resolved, base_path, location, parameter_name, required, required_declared, local_ref, warnings, xml_path=current_xml_path if location == 'body' else None)
    if normalized_type == 'array' and isinstance(resolved.get('items'), dict):
        item_schema = resolved['items']
        item_type, _ = normalize_type(item_schema, warnings, f'{base_path}[]')
        if item_type == 'object':
            walk_schema(item_schema, base_path + '[]', location, document, parameters, warnings, required=required, required_declared=required_declared, name=parameter_name, source_ref=local_ref, xml_path=current_xml_path)

# 원본 조회 (이후 저장 진행) : Swagger 조회 API에서 원본 OpenAPI JSON과 응답 메타데이터를 가져온다.
def fetch_swagger(url: str, timeout: int = REQUEST_TIMEOUT_SECONDS) -> tuple[dict[str, Any], dict[str, Any]]:
    request = urllib.request.Request(url, headers={'Accept': 'application/json', 'User-Agent': 'canonical-schema-preprocessor/1.0'})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        raw = response.read()
        response_meta = {'status': getattr(response, 'status', None), 'contentType': response.headers.get('Content-Type'), 'bytes': len(raw)}
    payload: Any = json.loads(raw.decode('utf-8-sig'))
    if isinstance(payload, str):
        payload = json.loads(payload)
    if not isinstance(payload, dict):
        raise ValueError('Swagger 응답이 JSON object가 아닙니다.')
    return payload, response_meta


#### 최종 문서/API 단위 Canonical Schema 생성 함수

In [3]:
# 하나의 OpenAPI operation을 CanonicalApi 객체로 변환한다.
# 대상 : API 단위
def build_canonical_api(document: dict[str, Any], document_title: str, endpoint: str, method: str, operation: dict[str, Any]) -> CanonicalApi:
    warnings: list[str] = []
    api_id = operation.get('operationId')
    if not api_id:
        fallback_endpoint = safe_filename(endpoint.strip('/').replace('/', '_'))
        api_id = f'{method.lower()}_{fallback_endpoint}'
        warnings.append('operationId가 없어 method와 endpoint로 apiId를 생성했습니다.')

    parameters: list[CanonicalParameter] = []
    for parameter in operation.get('parameters') or []:
        if not isinstance(parameter, dict):
            warnings.append(f'parameter가 object가 아님: {api_id}')
            continue
        location = parameter.get('in')
        name = parameter.get('name')
        if location not in {'header', 'path', 'query', 'cookie'} or not name:
            warnings.append(f'지원하지 않거나 이름이 없는 parameter: {api_id}')
            continue
        schema = parameter.get('schema') or {'type': 'string'}
        resolved = resolve_schema(schema, document, warnings)
        path_prefix = 'headers' if location == 'header' else location
        parameter_path = append_path(f'$.{path_prefix}', str(name))
        normalized_type, schema_format = normalize_type(resolved, warnings, parameter_path)
        xml = resolved.get('xml') if isinstance(resolved.get('xml'), dict) else {}
        parameters.append(CanonicalParameter(path=parameter_path, location=location, name=str(name), type=normalized_type, required=coerce_bool(parameter.get('required')), requiredDeclared='required' in parameter, description=parameter.get('description') or extract_description(resolved), format=schema_format, enum=resolved.get('enum'), example=resolved.get('example'), sourceRef=schema.get('$ref'), xmlName=xml.get('name') or str(name), xmlPath=None, xmlAttribute=coerce_bool(xml.get('attribute'))))

    request_body = operation.get('requestBody') or {}
    content = request_body.get('content') or {}
    content_type = next(iter(content), None)
    if len(content) > 1:
        warnings.append(f'여러 request content type 중 첫 번째만 사용: {sorted(content)}')
    if content_type:
        media = content.get(content_type) or {}
        schema = media.get('schema')
        if isinstance(schema, dict):
            walk_schema(schema, '$.body', 'body', document, parameters, warnings, required=coerce_bool(request_body.get('required')), required_declared='required' in request_body, source_ref=schema.get('$ref'))
        else:
            warnings.append('requestBody에 schema가 없습니다.')
    elif request_body:
        warnings.append('requestBody에 content가 없습니다.')

    info = document.get('info') or {}
    return CanonicalApi(documentId=DOCUMENT_ID, documentTitle=document_title, apiId=str(api_id), method=method.upper(), endpoint=endpoint, summary=operation.get('summary'), description=operation.get('description'), tags=[str(tag) for tag in operation.get('tags') or []], contentType=content_type, requestBodyRequired=coerce_bool(request_body.get('required')), parameters=parameters, source={'documentUrl': SWAGGER_URL, 'openapiVersion': document.get('openapi'), 'documentVersion': info.get('version')}, warnings=warnings)

# OpenAPI 문서의 모든 operation을 순회해 CanonicalDocument를 생성한다.
# 대상 : 전체 스웨거 문서
def build_canonical_document(document: dict[str, Any]) -> CanonicalDocument:
    openapi_version = document.get('openapi')
    if not isinstance(openapi_version, str) or not openapi_version.startswith('3.'):
        raise ValueError(f'지원하지 않는 OpenAPI 버전입니다: {openapi_version}')
    info = document.get('info') or {}
    document_title = str(info.get('title') or DOCUMENT_ID)
    paths = document.get('paths') or {}
    if not isinstance(paths, dict) or not paths:
        raise ValueError('Swagger paths가 비어 있습니다.')

    apis: list[CanonicalApi] = []
    seen_api_ids: set[str] = set()
    allowed_methods = {'get', 'post', 'put', 'patch', 'delete', 'head', 'options', 'trace'}
    for endpoint, path_item in paths.items():
        if not isinstance(path_item, dict):
            continue
        for method, operation in path_item.items():
            if method.lower() not in allowed_methods or not isinstance(operation, dict):
                continue
            api = build_canonical_api(document, document_title, endpoint, method, operation)
            if api.apiId in seen_api_ids:
                raise ValueError(f'문서 내 apiId 중복: {api.apiId}')
            seen_api_ids.add(api.apiId)
            apis.append(api)

    return CanonicalDocument(documentId=DOCUMENT_ID, documentTitle=document_title, openapiVersion=openapi_version, documentVersion=info.get('version'), sourceUrl=SWAGGER_URL, apis=apis)


#### 결과 저장 (JSON, SQLite)

In [4]:
# 원본 Swagger, API별 JSON, SQLite 전문, 품질 리포트를 저장한다.
def persist_results(raw_document: dict[str, Any], canonical_document: CanonicalDocument, fetch_meta: dict[str, Any]) -> dict[str, Any]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    API_JSON_DIR.mkdir(parents=True, exist_ok=True)
    raw_text = json.dumps(raw_document, ensure_ascii=False, indent=2)
    source_hash = hashlib.sha256(raw_text.encode('utf-8')).hexdigest()
    RAW_PATH.write_text(raw_text + '\n', encoding='utf-8')
    created_at = now_iso()
    api_dicts = [model_to_dict(api) for api in canonical_document.apis]

    for api in api_dicts:
        api_json = json.dumps(api, ensure_ascii=False, indent=2)
        api_path = API_JSON_DIR / f"{safe_filename(api['apiId'])}.json"
        api_path.write_text(api_json + '\n', encoding='utf-8')

    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS api_canonical_schema (
                document_id TEXT NOT NULL,
                api_id TEXT NOT NULL,
                document_title TEXT NOT NULL,
                method TEXT NOT NULL,
                endpoint TEXT NOT NULL,
                canonical_json TEXT NOT NULL,
                source_url TEXT NOT NULL,
                source_hash TEXT,
                created_at TEXT NOT NULL,
                updated_at TEXT NOT NULL,
                PRIMARY KEY (document_id, api_id)
            )
        """)
        for api in api_dicts:
            api_json = json.dumps(api, ensure_ascii=False, separators=(',', ':'))
            connection.execute("""
                INSERT INTO api_canonical_schema (document_id, api_id, document_title, method, endpoint, canonical_json, source_url, source_hash, created_at, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(document_id, api_id) DO UPDATE SET
                    document_title = excluded.document_title, method = excluded.method, endpoint = excluded.endpoint,
                    canonical_json = excluded.canonical_json, source_url = excluded.source_url, source_hash = excluded.source_hash,
                    updated_at = excluded.updated_at
            """, (api['documentId'], api['apiId'], api['documentTitle'], api['method'], api['endpoint'], api_json, SWAGGER_URL, source_hash, created_at, created_at))
        connection.commit()

    warnings_by_api = {api['apiId']: api['warnings'] for api in api_dicts if api['warnings']}
    report = {
        'documentId': canonical_document.documentId,
        'documentTitle': canonical_document.documentTitle,
        'sourceUrl': SWAGGER_URL,
        'sourceHash': source_hash,
        'fetch': fetch_meta,
        'generatedAt': created_at,
        'apiCount': len(api_dicts),
        'parameterCount': sum(len(api['parameters']) for api in api_dicts),
        'warningCount': sum(len(api['warnings']) for api in api_dicts),
        'warningsByApi': warnings_by_api,
        'rawPath': str(RAW_PATH),
        'databasePath': str(DB_PATH),
        'apiJsonDirectory': str(API_JSON_DIR),
    }
    REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return report


#### 실행

In [5]:
# 실행: 조회 API 호출 → Canonical Schema 생성 → API별 JSON/SQLite 저장
raw_swagger, fetch_meta = fetch_swagger(SWAGGER_URL)
canonical_document = build_canonical_document(raw_swagger)
report = persist_results(raw_swagger, canonical_document, fetch_meta)
report

{'documentId': 'hmall-openapi-v1',
 'documentTitle': 'HSHOP OPEN API Swagger',
 'sourceUrl': 'https://developers.hmall.com/main/swagger/5f205b21-baa0-44a6-a1ac-39c4d9395d38/1093902c-01a2-4167-8103-d0615749bdef?product_api_name=OPENAPI',
 'sourceHash': '9665d7bc4af5f99dccd66dfa808fd56e0f3fece5c3b132562a45ddf9a193e7b6',
 'fetch': {'status': 200,
  'contentType': 'application/json; charset=utf-8',
  'bytes': 237270},
 'generatedAt': '2026-10-24T08:08:13.721167+00:00',
 'apiCount': 43,
 'parameterCount': 908,
 'warningCount': 2,
 'warningsByApi': {'selectCnryCodeList': ['type 누락으로 string 처리: $.body',
   'type 누락으로 string 처리: $.body']},
 'rawPath': '/home/ubuntu/work/kosa-2026-hyundai-project/hyejung/mini_project_2/output/swagger_raw.json',
 'databasePath': '/home/ubuntu/work/kosa-2026-hyundai-project/hyejung/mini_project_2/output/canonical_schema.db',
 'apiJsonDirectory': '/home/ubuntu/work/kosa-2026-hyundai-project/hyejung/mini_project_2/output/canonical_schema/hmall-openapi-v1'}

In [6]:
# 저장 결과 확인
with sqlite3.connect(DB_PATH) as connection:
    rows = connection.execute(
        'SELECT document_id, api_id, method, endpoint, length(canonical_json), canonical_json '
        'FROM api_canonical_schema WHERE document_id = \'hmall-openapi-v1\' and api_id = \'selectItem\'' 
        'ORDER BY api_id LIMIT 10'
    ).fetchall()

rows

[('hmall-openapi-v1',
  'selectItem',
  'POST',
  '/pd/pdb/selectItem.do',
  2481,
  '{"documentId":"hmall-openapi-v1","documentTitle":"HSHOP OPEN API Swagger","apiId":"selectItem","method":"POST","endpoint":"/pd/pdb/selectItem.do","summary":"상품 상세 조회","description":"상품 상세 조회","tags":["PDBItemReg"],"contentType":"application/xml","requestBodyRequired":true,"parameters":[{"path":"$.headers.oauserId","location":"header","name":"oauserId","type":"string","required":true,"requiredDeclared":true,"description":null,"format":null,"enum":null,"example":null,"sourceRef":null,"xmlName":"oauserId","xmlPath":null,"xmlAttribute":false},{"path":"$.headers.oauseKey","location":"header","name":"oauseKey","type":"string","required":true,"requiredDeclared":true,"description":null,"format":null,"enum":null,"example":null,"sourceRef":null,"xmlName":"oauseKey","xmlPath":null,"xmlAttribute":false},{"path":"$.body.data1.id","location":"body","name":"id","type":"string","required":false,"requiredDeclared":fal

## 2번. Baseline 등록 및 검증

Canonical Schema SQLite에서 API를 조회한 후 request XML body를 검증하고, 오류가 없으면 baseline.db에 API별 Baseline을 저장한다. 입력은 마지막 셀의 BASELINE_DOCUMENT_ID, BASELINE_API_ID, BASELINE_XML_INPUT에 설정한다.


### 기본 설정 및 구조

In [7]:
# Baseline 등록·검증에 필요한 SQLite, XML, DataFrame 처리를 정의한다.
import xml.etree.ElementTree as ET
from decimal import Decimal, InvalidOperation

try:
    import pandas as pd
except ImportError:
    pd = None

BASELINE_DB_PATH = OUTPUT_DIR / 'baseline.db'
BASELINE_CONTENT_TYPE = 'application/xml'
BASELINE_ERROR_STATUSES = {
    'XML_STRUCTURE_ERROR', 'MISSING_FIELD', 'XML_ATTRIBUTE_ERROR',
    'TYPE_ERROR', 'ENUM_INVALID', 'UNKNOWN_FIELD',
}

# Baseline 검증 전체 요약의 표준 구조를 정의
class BaselineValidationSummary(BaseModel):
    document_id: str
    api_id: str
    validation_status: str
    total_fields: int = 0
    valid_count: int = 0    # 검증통과 필드 수
    required_empty_count: int = 0   # 검증에 넣지 않고, 따로 사용자 입력 받음
    error_count: int = 0
    unknown_field_count: int = 0

### XML 처리 (segment, xmlPath로 변환)

In [8]:
# empty 판정 : Baseline 값이 null, 빈 문자열 또는 공백인지 판정한다.
def is_empty_baseline_value(value: Any) -> bool:
    return value is None or (isinstance(value, str) and value.strip() == '')

# dataset id 추출 : Swagger example에서 괄호 앞 Dataset id만 추출한다.
def normalize_dataset_id(example: Any) -> str | None:
    if example is None:
        return None
    normalized = str(example).split('(', 1)[0].strip()
    return normalized or None

# namespace 제거 : XML namespace가 포함된 tag에서 로컬 요소명을 추출한다.
# ex> <ns:Root xmlns:ns="http://example.com"> => Root와 비교
def local_xml_name(tag: Any) -> str:
    text = str(tag)
    return text.rsplit('}', 1)[-1] if '}' in text else text

# Canonical Schema의 xml segment 변환 : Canonical xmlPath를 요소와 attribute segment 목록으로 파싱한다.
def parse_xml_path(xml_path: str) -> list[dict[str, Any]]:
    segments: list[dict[str, Any]] = []
    for part in [item for item in str(xml_path).strip('/').split('/') if item]:
        if part.startswith('@'):
            segments.append({'name': part[1:], 'attribute': True, 'selector': None})
            continue
        selector = re.fullmatch(r'(?P<name>[^\[]+)\[@(?P<attr>[^=]+)=(?P<quote>["\'])(?P<value>.*)(?P=quote)\]', part)
        if selector:
            segments.append({
                'name': selector.group('name'),
                'attribute': False,
                'selector': {'name': selector.group('attr'), 'value': selector.group('value')},
            })
        else:
            segments.append({'name': part, 'attribute': False, 'selector': None})
    return segments

# baseline과 canonical segment 요소명 비교 : XML element이 Canonical path segment와 일치하는지 확인한다.
def xml_element_matches(element: ET.Element, segment: dict[str, Any]) -> bool:
    if local_xml_name(element.tag) != segment['name']:
        return False
    selector = segment.get('selector')
    return selector is None or element.attrib.get(selector['name']) == selector['value']

# baseline에서 canonical schema에 맞는 요소 탐색하여 변환
# Canonical xmlPath에 해당하는 XML element 또는 attribute 값
# kind (element or attribute), element, value로 구성
def find_xml_path(root: ET.Element, segments: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if not segments or not xml_element_matches(root, segments[0]):
        return []
    current = [root]
    for segment in segments[1:]:
        if segment['attribute']:
            return [
                {'kind': 'attribute', 'element': element, 'value': element.attrib.get(segment['name'])}
                for element in current if segment['name'] in element.attrib
            ]
        current = [
            child
            for element in current
            for child in list(element)
            if xml_element_matches(child, segment)
        ]
    return [{'kind': 'element', 'element': element, 'value': element.text} for element in current]

# xml 요소를 segment로 변환 : XML element을 Dataset selector를 포함한 실제 경로 segment로 변환한다.
# ex> <Dataset id="customer01"> => Dataset[@id="customer01"]
def actual_xml_segment(element: ET.Element) -> str:
    name = local_xml_name(element.tag)
    if name == 'Dataset' and 'id' in element.attrib:
        return f'{name}[@id={json.dumps(element.attrib["id"], ensure_ascii=False)}]'
    return name

# xml 요소를 xmlPath로 변환 : XML 전체를 순회해 leaf element와 attribute의 실제 경로를 수집한다.
# actual_xml_segment 활용
# ex> /Root/Dataset[@id="customer01"]
def collect_actual_xml_paths(root: ET.Element) -> list[dict[str, Any]]:
    entries: list[dict[str, Any]] = []

    # XML tree 한 노드를 순회해 leaf와 attribute 경로를 누적한다.
    def visit(element: ET.Element, current_path: str) -> None:
        for attribute_name, value in element.attrib.items():
            entries.append({'path': f'{current_path}/@{attribute_name}', 'kind': 'attribute', 'name': attribute_name, 'value': value})
        children = list(element)
        if not children:
            entries.append({'path': current_path, 'kind': 'element', 'name': local_xml_name(element.tag), 'value': element.text})
            return
        for child in children:
            visit(child, f'{current_path}/{actual_xml_segment(child)}')

    visit(root, f'/{actual_xml_segment(root)}')
    return entries

### 검증 결과 생성 및 Canonical schema 조회

In [9]:
# 검증 결과 -> dictionary :검증 실패 또는 예외를 표준 결과 row로 만든다.
def make_validation_row(parameter: dict[str, Any] | None, status: str, message: str, actual_value: Any = None, path: str | None = None, xml_path: str | None = None) -> dict[str, Any]:
    parameter = parameter or {}
    return {
        'path': path or parameter.get('path'),
        'xmlPath': xml_path or parameter.get('xmlPath'),
        'name': parameter.get('name'),
        'xmlName': parameter.get('xmlName'),
        'expected_type': parameter.get('type'),
        'required': bool(parameter.get('required')),
        'expected_value': parameter.get('enum'),
        'actual_value': actual_value,
        'xmlAttribute': bool(parameter.get('xmlAttribute')),
        'status': status,
        'message': message,
    }

# Canonical Schema 조회 : Canonical Schema SQLite에서 document_id와 api_id에 해당하는 API 전문을 조회한다.
def load_canonical_api(document_id: str, api_id: str, db_path: Path | None = None) -> dict[str, Any]:
    if not document_id or not api_id:
        raise ValueError('document_id와 api_id는 필수입니다.')
    target_db = db_path or DB_PATH
    try:
        with sqlite3.connect(target_db, timeout=5) as connection:
            row = connection.execute(
                'SELECT canonical_json FROM api_canonical_schema WHERE document_id = ? AND api_id = ?',
                (document_id, api_id),
            ).fetchone()
    except sqlite3.OperationalError as exc:
        raise RuntimeError(f'Canonical Schema DB를 열 수 없습니다: {target_db}') from exc
    if row is None:
        raise KeyError(f'Canonical Schema를 찾을 수 없습니다: ({document_id}, {api_id})')
    result = json.loads(row[0])
    if not isinstance(result, dict):
        raise ValueError('canonical_json이 JSON object가 아닙니다.')
    return result

# xml 검증 : Baseline XML body와 Canonical Schema의 필드·구조·값을 검증한다.
# xml 기본구조 검증 : 빈값, 문법오류, content-type, root 요소
# Dataset ID 검증 : canonical에 없는 dataset
# Parameter 검증 : xmlPath, 데이터타입
def validate_baseline_xml(canonical_api: dict[str, Any], baseline_xml: str) -> dict[str, Any]:
    document_id = str(canonical_api.get('documentId') or canonical_api.get('document_id') or '')
    api_id = str(canonical_api.get('apiId') or canonical_api.get('api_id') or '')
    body_parameters = [p for p in canonical_api.get('parameters', []) if p.get('location') == 'body']
    rows: list[dict[str, Any]] = []

    if not baseline_xml or not baseline_xml.strip():
        rows.append(make_validation_row(None, 'XML_STRUCTURE_ERROR', 'Baseline XML body가 비어 있습니다.', path='$.body'))
        summary = BaselineValidationSummary(document_id=document_id, api_id=api_id, validation_status='FAIL', total_fields=len(body_parameters), error_count=1)
        return {'summary': summary.model_dump() if hasattr(summary, 'model_dump') else summary.dict(), 'rows': rows}

    try:
        root = ET.fromstring(baseline_xml)
    except ET.ParseError as exc:
        rows.append(make_validation_row(None, 'XML_STRUCTURE_ERROR', f'XML 문법 오류: {exc}', path='$.body'))
        summary = BaselineValidationSummary(document_id=document_id, api_id=api_id, validation_status='FAIL', total_fields=len(body_parameters), error_count=1)
        return {'summary': summary.model_dump() if hasattr(summary, 'model_dump') else summary.dict(), 'rows': rows}

    expected_content_type = str(canonical_api.get('contentType') or '').split(';', 1)[0].lower()
    if expected_content_type and expected_content_type != BASELINE_CONTENT_TYPE:
        rows.append(make_validation_row(None, 'XML_STRUCTURE_ERROR', f'지원하지 않는 contentType입니다: {expected_content_type}', path='$.body'))

    xml_parameters = [p for p in body_parameters if p.get('xmlPath')]
    expected_root_name = None
    if xml_parameters:
        root_segments = parse_xml_path(str(xml_parameters[0]['xmlPath']))
        expected_root_name = root_segments[0]['name'] if root_segments else None
    if expected_root_name and local_xml_name(root.tag) != expected_root_name:
        rows.append(make_validation_row(None, 'XML_STRUCTURE_ERROR', f'root 불일치: expected={expected_root_name}, actual={local_xml_name(root.tag)}', path='$.body'))

    expected_dataset_ids = {
        normalize_dataset_id(p.get('example'))
        for p in body_parameters
        if p.get('xmlAttribute') and normalize_dataset_id(p.get('example'))
    }
    actual_dataset_ids = {
        element.attrib.get('id')
        for element in root.iter()
        if local_xml_name(element.tag) == 'Dataset' and element.attrib.get('id') is not None
    }
    for actual_id in sorted(actual_dataset_ids - expected_dataset_ids):
        rows.append(make_validation_row(
            None, 'UNKNOWN_FIELD', f'Canonical Schema에 없는 Dataset id입니다: {actual_id}',
            actual_value=actual_id, path='$.body',
            xml_path=f'/Root/Dataset[@id={json.dumps(actual_id, ensure_ascii=False)}]',
        ))

    for parameter in body_parameters:
        xml_path = parameter.get('xmlPath')
        if not xml_path:
            rows.append(make_validation_row(parameter, 'XML_STRUCTURE_ERROR', 'Canonical Parameter에 xmlPath가 없습니다. Canonical Schema를 먼저 재생성하세요.'))
            continue
        matches = find_xml_path(root, parse_xml_path(str(xml_path)))
        if not matches:
            status = 'MISSING_FIELD' if parameter.get('required') else 'OPTIONAL_MISSING'
            rows.append(make_validation_row(parameter, status, 'XML에서 Canonical 경로를 찾지 못했습니다.'))
            continue

        values = [match['value'] for match in matches]
        non_empty_values = [value for value in values if not is_empty_baseline_value(value)]
        actual_value = values[0] if len(values) == 1 else values
        if parameter.get('required') and len(non_empty_values) != len(values):
            rows.append(make_validation_row(parameter, 'REQUIRED_EMPTY', 'required 필드에 빈 값이 있습니다.', actual_value=actual_value))
            continue

        status = 'VALID'
        messages: list[str] = []
        expected_type = str(parameter.get('type') or '').lower()
        if expected_type in {'integer', 'number'}:
            for value in non_empty_values:
                try:
                    numeric_value = Decimal(str(value).strip())
                    if not numeric_value.is_finite() or (expected_type == 'integer' and numeric_value != numeric_value.to_integral_value()):
                        raise InvalidOperation
                except (InvalidOperation, ValueError):
                    status = 'TYPE_ERROR'
                    messages.append(f'숫자 타입으로 변환할 수 없습니다: {value}')
                    break
        elif expected_type not in {'string', ''}:
            status = 'TYPE_CHECK_SKIPPED'
            messages.append(f'현재 단계에서 타입 검증을 생략합니다: {expected_type}')

        allowed_values = parameter.get('enum')
        if status in {'VALID', 'TYPE_CHECK_SKIPPED'} and allowed_values:
            allowed = {str(value) for value in allowed_values}
            invalid_values = [str(value) for value in non_empty_values if str(value) not in allowed]
            if invalid_values:
                status = 'ENUM_INVALID'
                messages.append(f'enum에 없는 값입니다: {invalid_values}')

        if not messages and not non_empty_values:
            messages.append('선택 필드의 값이 비어 있습니다.')
        rows.append(make_validation_row(parameter, status, ' '.join(messages) or '검증 통과', actual_value=actual_value))

    known_paths = {str(p.get('xmlPath')) for p in body_parameters if p.get('xmlPath')}
    for actual_entry in collect_actual_xml_paths(root):
        if actual_entry['path'] not in known_paths:
            rows.append(make_validation_row(
                None, 'UNKNOWN_FIELD', 'Canonical Schema에 정의되지 않은 XML element 또는 attribute입니다.',
                actual_value=actual_entry['value'], path='$.body', xml_path=actual_entry['path'],
            ))

    error_count = sum(row['status'] in BASELINE_ERROR_STATUSES for row in rows)
    required_empty_count = sum(row['status'] == 'REQUIRED_EMPTY' for row in rows)
    unknown_field_count = sum(row['status'] == 'UNKNOWN_FIELD' for row in rows)
    valid_count = sum(row['status'] == 'VALID' for row in rows)
    validation_status = 'FAIL' if error_count else ('PASS_WITH_REQUIRED_EMPTY' if required_empty_count else 'PASS')
    summary = BaselineValidationSummary(
        document_id=document_id, api_id=api_id, validation_status=validation_status,
        total_fields=len(body_parameters), valid_count=valid_count,
        required_empty_count=required_empty_count, error_count=error_count,
        unknown_field_count=unknown_field_count,
    )
    summary_dict = summary.model_dump() if hasattr(summary, 'model_dump') else summary.dict()
    return {'summary': summary_dict, 'rows': rows}

### 결과용 DF 변환 및 SQLite 저장

In [10]:
# DataFrame 결과 : Baseline 검증 결과를 DataFrame으로 변환한다.
def build_baseline_dataframe(validation_result: dict[str, Any]):
    if pd is None:
        raise ImportError('Baseline 결과를 DataFrame으로 표시하려면 pandas가 필요합니다.')
    columns = ['path', 'xmlPath', 'name', 'xmlName', 'expected_type', 'required', 'expected_value', 'actual_value', 'xmlAttribute', 'status', 'message']
    return pd.DataFrame(validation_result.get('rows', []), columns=columns)

# Baseline 저장 : 검증을 통과한 Baseline을 baseline.db에 API별 한 건으로 upsert한다.
def save_baseline(document_id: str, api_id: str, baseline_xml: str, canonical_api: dict[str, Any], validation_result: dict[str, Any], db_path: Path | None = None) -> bool:
    status = validation_result['summary']['validation_status']
    if status == 'FAIL':
        return False
    target_db = db_path or BASELINE_DB_PATH
    target_db.parent.mkdir(parents=True, exist_ok=True)
    timestamp = now_iso() if 'now_iso' in globals() else datetime.now(timezone.utc).isoformat()
    payload = json.dumps(validation_result, ensure_ascii=False, separators=(',', ':'))
    content_type = str(canonical_api.get('contentType') or BASELINE_CONTENT_TYPE)
    with sqlite3.connect(target_db, timeout=5) as connection:
        connection.execute('''
            CREATE TABLE IF NOT EXISTS api_baseline (
                document_id TEXT NOT NULL,
                api_id TEXT NOT NULL,
                baseline_xml TEXT NOT NULL,
                content_type TEXT NOT NULL,
                validation_status TEXT NOT NULL,
                validation_result_json TEXT NOT NULL,
                created_at TEXT NOT NULL,
                updated_at TEXT NOT NULL,
                PRIMARY KEY (document_id, api_id)
            )
        ''')
        connection.execute('''
            INSERT INTO api_baseline
            (document_id, api_id, baseline_xml, content_type, validation_status, validation_result_json, created_at, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(document_id, api_id) DO UPDATE SET
                baseline_xml = excluded.baseline_xml,
                content_type = excluded.content_type,
                validation_status = excluded.validation_status,
                validation_result_json = excluded.validation_result_json,
                updated_at = excluded.updated_at
        ''', (document_id, api_id, baseline_xml, content_type, status, payload, timestamp, timestamp))
        connection.commit()
    return True

# 최종 수행 함수 : Baseline을 조회하고 검증한 뒤 결과를 표시하고 저장한다.
def register_and_validate_baseline(document_id: str, api_id: str, baseline_xml: str, canonical_db_path: Path | None = None, baseline_db_path: Path | None = None) -> dict[str, Any]:
    canonical_api = load_canonical_api(document_id, api_id, canonical_db_path)
    validation_result = validate_baseline_xml(canonical_api, baseline_xml)
    dataframe = build_baseline_dataframe(validation_result)
    try:
        display(dataframe)
    except NameError:
        print(dataframe.to_string(index=False))
    saved = False
    if validation_result['summary']['validation_status'] != 'FAIL':
        saved = save_baseline(document_id, api_id, baseline_xml, canonical_api, validation_result, baseline_db_path)
    validation_result['saved'] = saved
    validation_result['dataframe'] = dataframe
    return validation_result

In [26]:
# Baseline 등록·검증에 사용할 document_id, api_id, XML body 입력값을 설정한다.
BASELINE_DOCUMENT_ID = DOCUMENT_ID
BASELINE_API_ID = 'selectItem'
BASELINE_XML_INPUT = """<?xml version="1.0" encoding="UTF-8"?>
<Root>
	<Dataset id="dsCond">
		<rows>
			<row>
				<slitmCd>2015700001</slitmCd>
				<venCd>001442</venCd>
			</row>
		</rows>
	</Dataset>
</Root>"""

#XML body를 입력한 뒤 아래 함수를 실행한다.
baseline_result = register_and_validate_baseline(
    document_id=BASELINE_DOCUMENT_ID,
    api_id=BASELINE_API_ID,
    baseline_xml=BASELINE_XML_INPUT,
)



,path,xmlPath,name,xmlName,expected_type,required,expected_value,actual_value,xmlAttribute,status,message
0,$.body.data1.id,"/Root/Dataset[@id=""dsCond""]/@id",id,id,string,False,None,dsCond,True,VALID,검증 통과
1,$.body.data1.data2.data3.slitmCd,"/Root/Dataset[@id=""dsCond""]/rows/row/slitmCd",slitmCd,slitmCd,string,False,None,2015700001,False,VALID,검증 통과
2,$.body.data1.data2.data3.itemCsfDCd,"/Root/Dataset[@id=""dsCond""]/rows/row/itemCsfDCd",itemCsfDCd,itemCsfDCd,string,False,None,None,False,OPTIONAL_MISSING,XML에서 Canonical 경로를 찾지 못했습니다.
3,$.body.data1.data2.data3.venCd,"/Root/Dataset[@id=""dsCond""]/rows/row/venCd",venCd,venCd,string,False,None,001442,False,VALID,검증 통과


In [27]:
# 저장 결과 확인
print(baseline_result['summary'])

with sqlite3.connect(BASELINE_DB_PATH) as connection:
    rows = connection.execute(
        'SELECT document_id, api_id, baseline_xml, validation_status, validation_result_json '
        'FROM api_baseline WHERE document_id = \'hmall-openapi-v1\' and api_id = \'selectItem\'' 
    ).fetchall()

rows

{'document_id': 'hmall-openapi-v1', 'api_id': 'selectItem', 'validation_status': 'PASS', 'total_fields': 4, 'valid_count': 3, 'required_empty_count': 0, 'error_count': 0, 'unknown_field_count': 0}


[('hmall-openapi-v1',
  'selectItem',
  '<?xml version="1.0" encoding="UTF-8"?>\n<Root>\n\t<Dataset id="dsCond">\n\t\t<rows>\n\t\t\t<row>\n\t\t\t\t<slitmCd>2015700001</slitmCd>\n\t\t\t\t<venCd>001442</venCd>\n\t\t\t</row>\n\t\t</rows>\n\t</Dataset>\n</Root>',
  'PASS',
  '{"summary":{"document_id":"hmall-openapi-v1","api_id":"selectItem","validation_status":"PASS","total_fields":4,"valid_count":3,"required_empty_count":0,"error_count":0,"unknown_field_count":0},"rows":[{"path":"$.body.data1.id","xmlPath":"/Root/Dataset[@id=\\"dsCond\\"]/@id","name":"id","xmlName":"id","expected_type":"string","required":false,"expected_value":null,"actual_value":"dsCond","xmlAttribute":true,"status":"VALID","message":"검증 통과"},{"path":"$.body.data1.data2.data3.slitmCd","xmlPath":"/Root/Dataset[@id=\\"dsCond\\"]/rows/row/slitmCd","name":"slitmCd","xmlName":"slitmCd","expected_type":"string","required":false,"expected_value":null,"actual_value":"2015700001","xmlAttribute":false,"status":"VALID","message":